In [1]:
!pip install transformers -q

In [1]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [2]:
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Max position: {model.config.n_positions}")
print(f"Device: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model: gpt2
Parameters: 124,439,808
Vocab size: 50257
Max position: 1024
Device: cuda


In [4]:
text = "Hello, I'm a unhappiness language model trained by OpenAI"

tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)
print(f"Text: {text}")
print(f"Tokens: {tokens}")
print(f"Token IDs: {ids}")
print(f"Number of tokens: {len(ids)}")

decoded = tokenizer.decode(ids)
print(f"Decoded: {decoded}")

Text: Hello, I'm a unhappiness language model trained by OpenAI
Tokens: ['Hello', ',', 'ĠI', "'m", 'Ġa', 'Ġunh', 'appiness', 'Ġlanguage', 'Ġmodel', 'Ġtrained', 'Ġby', 'ĠOpen', 'AI']
Token IDs: [15496, 11, 314, 1101, 257, 14274, 42661, 3303, 2746, 8776, 416, 4946, 20185]
Number of tokens: 13
Decoded: Hello, I'm a unhappiness language model trained by OpenAI


In [5]:
def greedy_generate(model, tokenizer, prompt, max_new_tokens=50):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    generated = input_ids
    print(f"generated type: {type(generated)}")
    print(f"generated shape: {generated.shape}")

    for i in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(generated) # outputs.logits shape: (1, seq_len, vocab_size)
            next_token_logits = outputs.logits[:, -1, :]
            next_token = next_token_logits.argmax(dim=-1, keepdim=True)
            if i == 0:
                print(f"outputs logits shape: {outputs.logits.shape}")
                print(f"next token: {next_token}")

            generated = torch.cat([generated, next_token], dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(generated[0], skip_special_tokens=True)

prompt = "The future of artificical intelligence is"
print(greedy_generate(model, tokenizer, prompt))

generated type: <class 'torch.Tensor'>
generated shape: torch.Size([1, 7])
outputs logits shape: torch.Size([1, 7, 50257])
next token: tensor([[287]], device='cuda:0')
The future of artificical intelligence is in the hands of the people.

The future of artificical intelligence is in the hands of the people.

The future of artificical intelligence is in the hands of the people.

The future of artificical intelligence is in the


In [7]:
tokens = tokenizer.tokenize("The future of artificical intelligence is")
print(tokens)
print(len(tokens))

['The', 'Ġfuture', 'Ġof', 'Ġartific', 'ical', 'Ġintelligence', 'Ġis']
7


In [9]:
def sample_generate(model, tokenizer, prompt, max_new_tokens=1000, temperature=1.0, top_k=0, top_p=1.0):
    """
    - temperature: 控制随机性(0.7-1.0)
    - top_k: 只从top-k个token里面采样(0=不限制)
    - top_p: 只从累计概率到p的tokens里面采样 (1.0=不限制)
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids

    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(generated)
            logits = outputs.logits[:, -1, :]

            # Temperature
            logits = logits/temperature

            # Top-k filtering
            if top_k > 0:
                top_k_values, _ = logits.topk(top_k)
                min_top_k = top_k_values[:, -1].unsqueeze(-1)
                logits = logits.masked_fill(logits < min_top_k, float('-inf'))

            if top_p < 1.0:
                sorted_logits, sorted_indicies = logits.sort(descending=True)
                cumulative_probs = sorted_logits.softmax(dim=-1).cumsum(dim=-1)
                # 找到累积概率超过 p 的位置，把它们设为 -inf
                sorted_mask = cumulative_probs - sorted_logits.softmax(-1) >=top_p
                sorted_logits[sorted_mask] = float('-inf')
                # 恢复原始顺序
                logits = sorted_logits.scatter(1, sorted_indicies, sorted_logits)

            probs = logits.softmax(dim=-1)
            next_token = torch.multinomial(probs, 1)
            generated = torch.cat([generated, next_token], dim=1)

            if next_token.item() == tokenizer.eos_token_id:
                break
        return tokenizer.decode(generated[0], skip_special_tokens=True)


In [8]:
prompt = "In a world where robots have become sentient,"

print("=" * 60)
print("🔵 GREEDY (repetitive, boring)")
print("=" * 60)
print(greedy_generate(model, tokenizer, prompt, max_new_tokens=80))

print("\n" + "=" * 60)
print("🟡 TEMPERATURE=0.3 (conservative)")
print("=" * 60)
print(sample_generate(model, tokenizer, prompt, temperature=0.3))

print("\n" + "=" * 60)
print("🟠 TEMPERATURE=1.0 (original distribution)")
print("=" * 60)
print(sample_generate(model, tokenizer, prompt, temperature=1.0))

print("\n" + "=" * 60)
print("🔴 TEMPERATURE=1.5 (very random, may be incoherent)")
print("=" * 60)
print(sample_generate(model, tokenizer, prompt, temperature=1.5))

print("\n" + "=" * 60)
print("🟢 TOP-K=50, TEMP=0.7 (balanced)")
print("=" * 60)
print(sample_generate(model, tokenizer, prompt, temperature=0.7, top_k=50))

print("\n" + "=" * 60)
print("🟣 TOP-P=0.9, TEMP=0.7 (best practice)")
print("=" * 60)
print(sample_generate(model, tokenizer, prompt, temperature=0.7, top_p=0.9))

🔵 GREEDY (repetitive, boring)
generated type: <class 'torch.Tensor'>
generated shape: torch.Size([1, 9])
outputs logits shape: torch.Size([1, 9, 50257])
next token: tensor([[340]], device='cuda:0')
In a world where robots have become sentient, it's hard to imagine a more important issue for the future of humanity than the future of the human race.

The future of humanity is not a question of whether we'll be able to survive, but of how we'll be able to survive.

The future of humanity is not a question of whether we'll be able to survive, but of how we'll be able to survive.

🟡 TEMPERATURE=0.3 (conservative)
In a world where robots have become sentient, we're going to have to start thinking about more of our own selves, and more of our own lives.

The question is, what do we do with our own selves?

It's not that we're not good at it.

We're not good at it.

We're not good at it.

We're not good at it.

We're not good at it.

We're not good at it.

We're not good at it.

We're not good

AttributeError: 'Tensor' object has no attribute 'topK'

In [10]:
print("\n" + "=" * 60)
print("🟢 TOP-K=50, TEMP=0.7 (balanced)")
print("=" * 60)
print(sample_generate(model, tokenizer, prompt, temperature=0.7, top_k=50))

print("\n" + "=" * 60)
print("🟣 TOP-P=0.9, TEMP=0.7 (best practice)")
print("=" * 60)
print(sample_generate(model, tokenizer, prompt, temperature=0.7, top_p=0.9))


🟢 TOP-K=50, TEMP=0.7 (balanced)
In a world where robots have become sentient, I'd hate to see a robot that's capable of making me think about anything other than my own life.

The real value of these new technologies is that they're not just a side effect of an already-existing AI. They're an opportunity to explore the possibilities of artificial intelligence in a way that is non-confrontational, non-threatening, and non-predictive. I'd love to see a robot that could understand the nuances of different kinds of situations (or maybe just understand the basic rules of the game). I'd love to have a robot that could understand a complex and complex world, that could understand the different kinds of people who live in different parts of the world, and that could see the difference between human and robot interactions.

Let's have a robot that is capable of seeing the world, but that can't understand what's going on in there, because they can't see anything other than what they're doing.



In [7]:
from transformers import set_seed
prompt = "The key to build great AI system is"
# input_ids = tokenizer.encode(prompt, return_tensors = 'pt').to(device)
tokenizer.pad_token = tokenizer.eos_token
# model.config.pad_token_id = tokenizer.eos_token_id

inputs = tokenizer(prompt, return_tensors="pt").to(device)


set_seed(42)
greedy_output = model.generate(inputs['input_ids'],
                               attention_mask=inputs['attention_mask'],
                               pad_token_id=tokenizer.eos_token_id,
                               max_new_tokens=80,
                               do_sample=False)
print("Greedy:", tokenizer.decode(greedy_output[0], skip_special_tokens=True))


set_seed(42)
sample_output = model.generate(inputs['input_ids'],
                               attention_mask=inputs['attention_mask'],
                               pad_token_id=tokenizer.eos_token_id,
                               max_new_tokens=80,
                               do_sample=True,
                               temperature=0.7,
                               top_p=0.9)
print("\n\nSampling:", tokenizer.decode(sample_output[0], skip_special_tokens=True))

set_seed(42)
beam_output = model.generate(
    inputs['input_ids'],
    attention_mask=inputs['attention_mask'],
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=80,
    num_beams=5,
    no_repeat_ngram_size=2,
    early_stopping=True,
)
print("\n\nBeam:", tokenizer.decode(beam_output[0], skip_special_tokens=True))

Greedy: The key to build great AI system is to have a good understanding of the world around you. This is especially important when you are working with a large number of people.

The key to building great AI system is to have a good understanding of the world around you. This is especially important when you are working with a large number of people. The key to build great AI system is to have a good understanding of the world around you


Sampling: The key to build great AI system is to have good communication between the AI and the AI. When you have communication between the AI and the AI, it will be very helpful for the AI to understand what is going on and how to improve the system.

Let's start with the AI and the communication between the two.

The AI has to understand a lot about the language and it needs to be able to understand a lot


Beam: The key to build great AI system is to be able to predict what is going to happen in the future.

In this article, I will show you how y

In [4]:
print(model.config)
print()

for name, parameter in model.named_parameters():
    if 'h.' not in name:
        print(f"{name}: {parameter.shape}")
    if 'h.0.' in name:
        print(f"{name}: {parameter.shape}")

GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "use_cache": true,
  "vocab_size": 50257
}


transformer.wte.weight: tor